# Pockets and paths in IRMOF-1

In this example, we analyse the cavities of IRMOF-1, to identify the different types of pockets present using the characteristic geometric size distribution. Further, we represent these properties and connectivity of these pockets as a pore graph, and detect all the paths that run along the length of the unit cell in any direction. The paths are then separated into 1D vs higher dimensional, and compared amongst each other, for uniqueness. Paths are isomorphic only if they connect same types of regions in the same order.

## Read the cif file
As usual we read in the framework configuration use ASE, although we are using a CIF file in this example, any format supported by ASE should work.

- We then update the ase atoms object covalent bonds
- Create a structure graph from atoms and bonds

The last two steps are optional, but are quite useful for generating more sensible visualizations of the framework.

In [ ]:
#region #* Read cif and compute bonds
import mofography as mgr
from ase.io import read
# Reading the cif file and storing it in a variable called mat_atoms.
# mat_atoms = read('./TestCIFs/LTA96.cif')
# mat_atoms = read('./TestCIFs/NU-1000_S1.cif')
# mat_atoms = read('./TestCIFs/NU-1000_S1.cif')
# mat_atoms = read('./TestCIFs/ZIF-8-RASPA.cif')
# mat_atoms = read('./TestCIFs/P1-Cu-BTC.cif')
mat_atoms = read('../examples/TestCIFs/P1-IRMOF-1.cif')
# mat_atoms = read('./TestCIFs/MIL-47-P1-RASPA.cif')
# mat_atoms = read('./TestCIFs/VUSKEA_clean.cif')
# mat_atoms = read('./TestCIFs/hypotheticalMOF_1003830_0_0_4_10_10_0.cif')
# mat_atoms = read('./TestCIFs/hypotheticalMOF_2000756_0_0_2_32_32_0.cif')
# mat_atoms = read('./TestCIFs/hypotheticalMOF_5026325_3_0_0_27_27_0.cif')
# mat_atoms = read('./TestCIFs/hypotheticalMOF_7002752_0_0_4_28_28_0.cif')
# mat_atoms = read('./TestCIFs/ADARAA_clean.cif')
# mat_atoms = read('./TestCIFs/ZIF-69-RASPA.cif')
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)
mat_graph = mgr.sgraph_from_atoms(mat_updated)
#endregion


## Compute the distance grid

We are computing a 0.25 $\AA$ spaced distance grid using a point probe, as a starting point for our analysis. In this example, the covalent radii from ASE are used as framework atom radii. You may use any other list of radii as you please. For more information one the compute options, please refer to the API documentation.

In [ ]:
#region #* compute the distance grid
spacing=0.25
import mofography as mgr
from ase.data import covalent_radii

radii = covalent_radii[mat_updated.get_atomic_numbers()]
grid = mgr.dgrid_from_atoms(mat_atoms, radii=radii, spacing =spacing, block_size=20000)
#endregion

## Compute the region segmentation

In [ ]:
##region #* Compute region segmentation and local maxima
regions, maxima = mgr.regions_from_dgrid(grid.compute())#, mask_thickness=1.5,exclude_border=False, min_distance=10 , threshold_abs=3, p_norm=2)
#endregion

## Compute the RAG 
RAG stands for Region Adjacency Graph (RAG) a.k.a the Pore Graph

In [ ]:
#region #* Compute the poregraph
rag2 = mgr.get_rag_from_regions_and_grid(regions, grid.compute(), mat_updated,maxima)
#endregion

## Compute PBC 
We represmt the Periodic Boudnary Conditions in two differnet ways:
1. As groups of regions that are one and the same in the context of PBC

In this case, for example, all 8 regions on the corners form one group, all 4 regions on the edges parallel to say, the y axis form a group, 2 regions that lie on opposing faces form a periodic group. In IRMOF-1 where the unit cell cuts through the cavities in a symmetric way, each group when put together forms a complete cage.

2. As a dictionary mapping from old region indices to PBC group index

This contains exactly the same informaton as before, but, provides a convenient way of applying PBC to large numpy arrays.

In [ ]:
#region #* Useful PBC information in groups and as mapping
pbc_groups = mgr.get_periodic_map(regions=regions, maxima=maxima, maxima_radii=grid.compute()[tuple(maxima.T)], return_conn=True, wall_thickness=4)
pmap= mgr.get_periodic_map(regions=regions, maxima=maxima, maxima_radii=grid.compute()[tuple(maxima.T)], return_conn=False, wall_thickness=4)
#endregion

## Make nice plot traces

Let's make some nice Plotly figure traces for what we have computed so far, so that we can plot things later. 
1. Scatter 3d trace for the region adjacency graph
2. A 2.0 $\AA$ isosurface to represnet the distance grid
3. The unit cell as a 3D lines (also a scatter 3D )

In [ ]:
#region# * Make traces for surface, pore graph, unit cell
rag_trace=mgr.plot_rag(rag2,plot_with='maxima', name='p-graph') 
# surface_trace=plot_surface_dgrid2(grid.compute(), 2.0, select=list(pbc_groups[0].union(pbc_groups[2].union(pbc_groups[-3]))),ase_atoms=mat_updated,region_labels=regions, stylize=True)
surface_trace=mgr.plot_surface_dgrid(grid.compute(), 2.0, select='all',ase_atoms=mat_updated,region_labels=regions, stylize=True)
cell_trace=mgr.plot_cell(mat_updated)
frame_trace = mgr.plot_framework_as_mesh(mat_updated, show_rings=False, stylize=True)
#endregion

In [ ]:
#region # * Plot the figure with all the traces
# fig = go.Figure(data=[*frame_trace, cell_trace, surface_trace, rag_trace]).update(layout=mgr.dark_layout)
# fig.show()
#endregion

## Node level features
We compute some properties of hte regions to distinguish the differnet typs of pockets in IRMOF-1. We annotate these properties on to the poe graph for convenience.


    Note: We can see the two kinds of pockets in the visualization, this might seem non intuitive in the case of IRMOF-1. This happens because of the orienation of the aromatic rings in the opposing BDC linkers. They are not perfectly paralle, instead, are oriented inward (or outward depending where ou look at it from), making some pockets smaller than the others. 

In [ ]:
#region #* Let's add some properties to the pore graph (rag) and visualize
from skimage.measure import regionprops
props = regionprops(regions)
# * size dist about centroid with and without pbc 
rag2 = mgr.add_psd_to_rag(rag2,props,  bins=np.linspace(0,30,50), ase_atoms=mat_updated, shape = regions.shape)
rag2 = mgr.add_psd_to_rag_with_pbc(rag=rag2,props =props, pbc_groups = pbc_groups, bins=np.linspace(0,30,50), ase_atoms=mat_updated, shape = regions.shape)
fig1=go.Figure() 
for group in pbc_groups:
    node = list(group)[0]
    fig1.add_trace(go.Scatter(x=rag2.nodes[node]['psd_x_pbc'], y=rag2.nodes[node]['psd_y_pbc'], mode='markers+lines', name=str(rag2.nodes[node]['labels'][0])))
# pio.renderers.default = "notebook_connected"
fig1.update_layout(font_family='arial', title=dict(text='Size dist. of regions (one per PBC group)'), xaxis=dict(title='Cavity size (\u212b)'), yaxis=dict(title='Density histogram')).show()
# fig1.show()
#
#endregion

## Find Paths

We find all the paths in the material, that run along the whole lenght of the unit cell, regardless of any particular direction

    Note 1: Only the shortest path between a pair of nodes is considered, as there probably a large number of circuitous paths between a pair of nodes in each graph.

    Note 2: If two paths are of the exact same length, only one of them is shown, since these paths are often symmetric. You can use networkx to find all the paths between two nodes using find_shortest_paths function.

We then separate the paths out into 1D and higher dimensional using PCA, as 1D paths are perhaps more interesting for practical purposes. 

    Note 3: 1D path does not refer to teh connectivity of the pockets, it simply means all the region maxima that form the path lie on a fairly straight line, as set by the pca threshold of the leading principal direction

In [ ]:
#region #* Let's find the unique 1D and 2D channels in the MOF
pca_list, paths_list= mgr.find_paths(regions, rag2)

# * separate into 1D and 2D paths
threshold = 0.99 #* This is our strict criterio for something to be 1D
paths1d_to_plot = np.array(paths_list)[np.where([check[0]>threshold for check in pca_list])[0]].tolist()
pathshd_to_plot = np.array(paths_list)[np.where([check[0]<threshold for check in pca_list])[0]].tolist()
#
checks1d= np.array(pca_list)[np.where([check[0]>threshold for check in pca_list])[0]].tolist()
checkshd   = np.array(pca_list)[np.where([check[0]<threshold for check in pca_list])[0]].tolist()
#endregion


## Find unique paths

Find unique paths among the list of 1D and higher D paths using graph isomorphism, where paths that connect the same type of regions in the same order are considered non-unique. 

In [ ]:
#region #* # Finding the isomorphic path groups. (paths that are truly different) 
path_groups_1d  = [list(p)[0] for p in mgr.find_isomorphic_path_groups(paths1d_to_plot, rag2)]
path_groups_hd  = [list(p)[0] for p in mgr.find_isomorphic_path_groups(pathshd_to_plot, rag2)]


## Visualize Everything

Finally, we make some more traces for paths, and take the rest of the traces we computed earlier, and make a nice, interactive plot using Plotly.

Note: If the plot is too slow, double click on traces to isolate, single click to toggle. Usually the mesh traces are RAM intensive. You can also reduce the stepsize of the isosurface calculation to make a coarser but more responsive trace.

In [ ]:
#region #* Finally let's visualize everything
# path_groups_hd = find_isomorphic_path_groups(pathshd_to_plot, rag2)
import plotly.graph_objects as go 
import plotly.io as pio 
# pio.renderers.default='notebook_connected'
paths_1d_trace = mgr.plot_paths(rag2, path_groups_1d, legendgroup='1', legendgrouptitle_text='1D Channels')
paths_hd_trace= mgr.plot_paths(rag2,path_groups_hd, legendgroup='2', legendgrouptitle_text='2D & 3D Channels')
fig = go.Figure(data=[*frame_trace, *paths_1d_trace, *paths_hd_trace,cell_trace, surface_trace, rag_trace]).update(layout=mgr.white_layout)
fig.show()
#
#endregion